# 01 — PDF/HTML Extraction & Section Splitting

**Goal of this notebook:** take the raw downloaded filings (4 EDGAR `.htm` files + 1 Infosys `.pdf`),
extract clean text, pull out the MD&A and Risk Factors sections for each company, and **save them to
disk** as `.txt` files.

Why save to disk instead of just keeping variables in memory? Because every other notebook
(02, 03, 04, 05) will **load these `.txt` files fresh** instead of depending on this notebook's
variables still being in memory. This avoids the `NameError` / kernel-state confusion from before —
each notebook becomes independently runnable.

## Setup — install/import required libraries

In [13]:
import os

PROJECT_ROOT = r"C:\Users\Devanshi\financial-nlp-analyzer"
os.chdir(PROJECT_ROOT)
print("Working directory set to:", os.getcwd())

Working directory set to: C:\Users\Devanshi\financial-nlp-analyzer


In [14]:
import re
import os
import pdfplumber
from bs4 import BeautifulSoup

# Folder structure (matches Phase 4)
PROJECT_ROOT = r"C:\Users\Devanshi\financial-nlp-analyzer"  # edit to your actual folder
RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw_pdfs")
EXTRACTED_DIR = os.path.join(PROJECT_ROOT, "data", "extracted_text")
os.makedirs(EXTRACTED_DIR, exist_ok=True)


## Company file map — EDIT THIS if your filenames are different

In [15]:
# filename must match exactly what's in your data/raw_pdfs/ folder
COMPANIES = {
    "jpmorgan":   {"file": "jpm_10k_2024.htm",        "type": "html"},
    "caterpillar":{"file": "caterpillar_10k_2024.htm","type": "html"},
    "accenture":  {"file": "accenture_10k_2024.htm",  "type": "html"},
    "pg":         {"file": "pg_10k_2024.htm",         "type": "html"},
    "infosys":    {"file": "infosys-ar-25.pdf",     "type": "pdf"},   # rename to match your actual file
}


## Extraction functions

In [16]:
def extract_text_from_pdf(filepath):
    """Extracts all text from a PDF, page by page."""
    full_text = []
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text.append(text)
    return "\n".join(full_text)


def extract_text_from_html(filepath):
    """Extracts clean text from an EDGAR 10-K HTML file."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    return "\n".join(lines)


def clean_text(text):
    """Strips common PDF/HTML junk: dot-leaders, page numbers, excess whitespace."""
    text = re.sub(r'\.{4,}', ' ', text)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()


def extract_section(full_text, start_markers, end_markers, min_length=500):
    def find_all(markers):
        positions = []
        for m in markers:
            pattern = re.escape(m).replace(r'\ ', r'\s+')  # tolerate wrapped/extra whitespace
            positions += [x.start() for x in re.finditer(pattern, full_text, re.IGNORECASE)]
        return sorted(positions)

    start_positions = find_all(start_markers)
    end_positions = find_all(end_markers)

    for s in start_positions:
        later_ends = [e for e in end_positions if e > s]
        if not later_ends:
            continue
        e = min(later_ends)
        if (e - s) > min_length:
            return full_text[s:e].strip()
    return None



## Extract & split every company — EDGAR (Item-numbered) companies

For US 10-Ks, sections are standardized:
- Risk Factors = Item 1A → Item 1B/Item 2
- MD&A = Item 7 → Item 7A/Item 8

In [17]:
import os
print("Current working directory:", os.getcwd())
print()
print("Does data/raw_pdfs exist here?", os.path.exists("data/raw_pdfs"))
print()
if os.path.exists("data/raw_pdfs"):
    print("Files found:", os.listdir("data/raw_pdfs"))

Current working directory: C:\Users\Devanshi\financial-nlp-analyzer

Does data/raw_pdfs exist here? True

Files found: ['accenture_10k_2024.htm', 'caterpillar_10k_2024.htm', 'infosys-ar-25.pdf', 'jpm_10k_2024.htm', 'pg_10k_2024.htm']


In [18]:
edgar_companies = ["jpmorgan", "caterpillar", "accenture", "pg"]
extracted = {}

for name in edgar_companies:
    filepath = os.path.join(RAW_DIR, COMPANIES[name]["file"])
    print(f"--- {name} ---")

    raw_text = extract_text_from_html(filepath)
    raw_text = clean_text(raw_text)

    risk = extract_section(
        raw_text,
        start_markers=["Item 1A.", "Item 1A", "ITEM 1A."],
        end_markers=["Item 1B.", "Item 1B", "ITEM 1B.", "Item 2."]
    )
    mdna = extract_section(
        raw_text,
        start_markers=["Item 7.", "Item 7", "ITEM 7."],
        end_markers=["Item 7A.", "Item 8.", "ITEM 7A."]
    )

    print(f"Risk Factors: {len(risk) if risk else 'NOT FOUND'} chars")
    print(f"MD&A:         {len(mdna) if mdna else 'NOT FOUND'} chars")
    extracted[name] = {"full": raw_text, "mdna": mdna, "risk": risk}

--- jpmorgan ---
Risk Factors: 136029 chars
MD&A:         NOT FOUND chars
--- caterpillar ---
Risk Factors: 75113 chars
MD&A:         209867 chars
--- accenture ---
Risk Factors: 91511 chars
MD&A:         46256 chars
--- pg ---
Risk Factors: 38613 chars
MD&A:         91326 chars


## VALIDATE before moving on

For each company, check the printed lengths above. If any section says `NOT FOUND`, or the length
looks suspiciously small (under ~1000 characters), print a sample and inspect it — don't proceed
to notebook 02 with broken sections.

In [19]:
for name in edgar_companies:
    mdna = extracted[name]["mdna"]
    risk = extracted[name]["risk"]
    print(f"=== {name} ===")
    print("MD&A sample:", (mdna[:200] if mdna else "MISSING"))
    print("Risk sample:", (risk[:200] if risk else "MISSING"))
    print()


=== jpmorgan ===
MD&A sample: MISSING
Risk sample: Item 1A. Risk Factors.
The following discussion sets forth the material risk factors that could affect JPMorganChase’s financial condition and operations. Readers should not consider any descriptions 

=== caterpillar ===
MD&A sample: Item 7 “Management’s Discussion and Analysis of Financial Condition and Results of Operations.”
Construction Industries
Our Construction Industries segment is primarily responsible for supporting cust
Risk sample: Item 1A. of this Form 10-K.
In managing foreign currency risk for Cat Financial’s operations, the objective is to minimize earnings volatility resulting from conversion and the remeasurement of net fo

=== accenture ===
MD&A sample: Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations
Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations
The followin
Risk sample: Item 1A. Risk Factors
Item 1A. Risk Factors
In additi

## Infosys (PDF, non-standardized) — needs manual section-finding

Infosys doesn't use "Item 1A" / "Item 7" numbering. We search for the actual heading text instead.
**This will likely need tweaking based on what your specific PDF's headings look like — inspect the
printed samples and adjust `start_markers`/`end_markers` if needed.**

In [20]:
infosys_path = os.path.join(RAW_DIR, COMPANIES["infosys"]["file"])
infosys_full = extract_text_from_pdf(infosys_path)
infosys_full = clean_text(infosys_full)
print(f"Infosys total extracted: {len(infosys_full):,} characters")

# Try common heading phrasing — adjust these if NOT FOUND
infosys_mdna = extract_section(
    infosys_full,
    start_markers=["Management's Discussion and Analysis", "Management Discussion and Analysis"],
    end_markers=["Business Responsibility", "Corporate Governance", "Risk Management Report"],
    min_length=1000
)
infosys_risk = extract_section(
    infosys_full,
    start_markers=["Risk Management", "Risk Management Report"],
    end_markers=["Corporate Governance", "Business Responsibility"],
    min_length=1000
)

print("MD&A:", len(infosys_mdna) if infosys_mdna else "NOT FOUND — inspect infosys_full manually")
print("Risk:", len(infosys_risk) if infosys_risk else "NOT FOUND — inspect infosys_full manually")

extracted["infosys"] = {"full": infosys_full, "mdna": infosys_mdna, "risk": infosys_risk}


Infosys total extracted: 1,166,131 characters
MD&A: 5092
Risk: 25453


## If Infosys sections weren't found — manual inspection helper

Run this to print every line that looks like a heading (short, title-like) so you can find the
actual wording used in your specific PDF, then update the markers above and re-run.

In [ ]:
# Uncomment and run if infosys_mdna or infosys_risk is None
# lines = infosys_full.split("\n")
# candidate_headings = [l for l in lines if 5 < len(l) < 80 and l.strip()[0:1].isupper()]
# for h in candidate_headings[:60]:
#     print(h)


## Save everything to disk — this is what notebooks 02-05 will load

In [21]:
for name, sections in extracted.items():
    for section_name, text in sections.items():
        if text is None:
            continue
        out_path = os.path.join(EXTRACTED_DIR, f"{name}_{section_name}.txt")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(text)
        print(f"Saved {out_path} ({len(text):,} chars)")

Saved C:\Users\Devanshi\financial-nlp-analyzer\data\extracted_text\jpmorgan_full.txt (1,403,161 chars)
Saved C:\Users\Devanshi\financial-nlp-analyzer\data\extracted_text\jpmorgan_risk.txt (136,029 chars)
Saved C:\Users\Devanshi\financial-nlp-analyzer\data\extracted_text\caterpillar_full.txt (536,677 chars)
Saved C:\Users\Devanshi\financial-nlp-analyzer\data\extracted_text\caterpillar_mdna.txt (209,867 chars)
Saved C:\Users\Devanshi\financial-nlp-analyzer\data\extracted_text\caterpillar_risk.txt (75,113 chars)
Saved C:\Users\Devanshi\financial-nlp-analyzer\data\extracted_text\accenture_full.txt (390,793 chars)
Saved C:\Users\Devanshi\financial-nlp-analyzer\data\extracted_text\accenture_mdna.txt (46,256 chars)
Saved C:\Users\Devanshi\financial-nlp-analyzer\data\extracted_text\accenture_risk.txt (91,511 chars)
Saved C:\Users\Devanshi\financial-nlp-analyzer\data\extracted_text\pg_full.txt (325,410 chars)
Saved C:\Users\Devanshi\financial-nlp-analyzer\data\extracted_text\pg_mdna.txt (91,326

In [ ]:
#dont run
import re
candidates = ["Item 1B", "Item 2", "PART II"]
for marker in candidates:
    positions = [m.start() for m in re.finditer(re.escape(marker), accenture_text, re.IGNORECASE)]
    print(f"{marker}: {len(positions)} matches")
    if positions:
        print(accenture_text[positions[0]-40:positions[0]+100], "\n---")

Item 1B: 0 matches
Item 2: 0 matches
PART II: 7 matches
ponse to Items 10, 11, 12, 13 and 14 of Part III. The definitive proxy statement will be filed with the SEC not later than 120 days after th 
---


In [ ]:
#dont run
import re

# Load Accenture's raw extracted text (re-run notebook 01's extraction for Accenture if needed)
accenture_path = "data/raw_pdfs/accenture_10k_2024.htm"
accenture_text = extract_text_from_html(accenture_path)
accenture_text = clean_text(accenture_text)

matches = [m.start() for m in re.finditer(r'Item\s*1A', accenture_text, re.IGNORECASE)]
print(f"Found {len(matches)} occurrences of 'Item 1A'")
for m in matches[:10]:
    print("---")
    print(accenture_text[m:m+100])

Found 18 occurrences of 'Item 1A'
---
Item 1A.
Risk Factors
Item 1B.
Unresolved Staff Comments
Item 1C.
Cybersecurity
Item 2.
Properties
I
---
Item 1A. Risk Factors
Item 1A. Risk Factors
In addition to the other information set forth in this r
---
Item 1A. Risk Factors
In addition to the other information set forth in this report, you should care
---
Item 1A. Risk Factors
engineering and manufacturing, and robotics solutions. As we expand our servic
---
Item 1A. Risk Factors
or regulatory scrutiny, litigation or other legal liability, or ethical concer
---
Item 1A. Risk Factors
We face legal, reputational and financial risks from any failure to protect cl
---
Item 1A. Risk Factors
•
accounting firms and consultancies that provide consulting, managed services
---
Item 1A. Risk Factors
If we do not successfully manage and develop our relationships with key ecosys
---
Item 1A. Risk Factors
Our profitability could suffer if our cost-management strategies are unsuccess
---
Item 1A. Risk Fac

In [ ]:
#dont run
accenture_risk = extract_section(
    accenture_text,
    start_markers=["Item 1A. Risk Factors", "Item 1A"],
    end_markers=["Item 1B", "Item 2", "PART II"],
)

In [ ]:
#dont run
import os
print("Current working directory:", os.getcwd())

Current working directory: C:\Users\Devanshi\financial-nlp-analyzer


In [ ]:
#dont run
accenture_path = os.path.join(RAW_DIR, COMPANIES["accenture"]["file"])
accenture_text = extract_text_from_html(accenture_path)
accenture_text = clean_text(accenture_text)

accenture_risk = extract_section(
    accenture_text,
    start_markers=["Item 1A.", "Item 1A", "ITEM 1A."],
    end_markers=["Item 1B.", "Item 1B", "ITEM 1B.", "Item 2."]
)

print("Type:", type(accenture_risk))
print("Length:", len(accenture_risk) if accenture_risk else "STILL NONE")

Type: <class 'NoneType'>
Length: STILL NONE


In [ ]:
#dont run
import inspect
print(inspect.getsource(extract_section))

def extract_section(full_text, start_markers, end_markers, min_length=500):
    """
    Finds ALL occurrences of start/end markers and tries every valid
    (start, nearest-following-end) pair, returning the first one long
    enough to be the real section (not a Table of Contents entry).
    """
    for start in start_markers:
        start_positions = [m.start() for m in re.finditer(re.escape(start), full_text, re.IGNORECASE)]
        for end in end_markers:
            end_positions = [m.start() for m in re.finditer(re.escape(end), full_text, re.IGNORECASE)]
            for s in start_positions:
                later_ends = [e for e in end_positions if e > s]
                if not later_ends:
                    continue
                e = min(later_ends)
                if (e - s) > min_length:
                    return full_text[s:e].strip()
    return None

